## Extract safe pairs from MoNA dataset to perform Active Learning

In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from tqdm import tqdm
import os

In [2]:
# --- 1. CONFIGURATION ---

# A. ORIGINAL UNIVERSE (Your current Train/Val/Test)
# These files define the IDs used in your existing splits
ORIG_MOL_DF = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/mol_df.pkl"
ORIG_SPEC_DF = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/spec_df.pkl"
SPLIT_DIR = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits"

In [19]:
# B. MONA UNIVERSE (The New Data)
# These files define the IDs used in the new candidate pairs
MONA_MOL_DF = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/mol_df_mona.pkl" # Update if name differs
MONA_SPEC_DF = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/spec_df_mona.pkl"
NEW_CANDIDATES = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/spectra_pairs/balanced_10bin_mona_dataset.feather"

In [23]:
OUTPUT_PATH = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/mass_gate_2_percent/large_safe_augmentation_candidates.feather"

In [37]:
# --- 2. HELPER: MAP BUILDER ---
def build_lookup_maps(mol_df_path, spec_df_path, name="Dataset"):
    print(f"\n--- Building Lookups for {name} ---")
    
    # 1. Mol ID -> Scaffold
    print(f"Loading Molecules: {mol_df_path}")
    df_mols = pd.read_pickle(mol_df_path)
    # Ensure scaffold column exists
    if 'scaffold' not in df_mols.columns:
        raise ValueError(f"'{mol_df_path}' is missing the 'scaffold' column!")
    
    mol2scaffold = dict(zip(df_mols['mol_id'], df_mols['scaffold']))
    
    # 2. Spec ID -> Mol ID & Precursor m/z
    print(f"Loading Spectra: {spec_df_path}")
    df_specs = pd.read_pickle(spec_df_path)
    
    spec2mol = dict(zip(df_specs['spec_id'], df_specs['mol_id']))
    spec2mz = dict(zip(df_specs['spec_id'], df_specs['prec_mz']))
    
    print(f" > {name} Ready: {len(spec2mol)} spectra, {len(mol2scaffold)} molecules.")
    return spec2mol, mol2scaffold, spec2mz

In [26]:
# --- 3. BUILD ORIGINAL FIREWALL ---
# We use the ORIGINAL maps to decode the Validation/Test sets
orig_spec2mol, orig_mol2scaffold, _ = build_lookup_maps(ORIG_MOL_DF, ORIG_SPEC_DF, "ORIGINAL DB")

print("\n--- Building Scaffold Firewall from Original Splits ---")
val_path = os.path.join(SPLIT_DIR, "stratified_binary_07_dataset_val.feather")
test_path = os.path.join(SPLIT_DIR, "stratified_binary_07_dataset_test.feather")
df_val = pd.read_feather(val_path)
df_test = pd.read_feather(test_path)


--- Building Lookups for ORIGINAL DB ---
Loading Molecules: /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/mol_df.pkl
Loading Spectra: /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/spec_df.pkl
 > ORIGINAL DB Ready: 227308 spectra, 27969 molecules.

--- Building Scaffold Firewall from Original Splits ---


In [27]:
# Gather all forbidden Spec IDs
forbidden_specs = set(df_val['name_main']) | set(df_val['name_sub']) | \
                  set(df_test['name_main']) | set(df_test['name_sub'])

# Convert to Forbidden Scaffolds
forbidden_scaffolds = set()
for spec_id in tqdm(forbidden_specs, desc="Mapping Firewall"):
    mol_id = orig_spec2mol.get(spec_id)
    if mol_id is not None:
        scaff = orig_mol2scaffold.get(mol_id)
        if scaff:
            forbidden_scaffolds.add(scaff)

print(f" > FIREWALL ACTIVE: {len(forbidden_scaffolds)} unique scaffolds blocked.")

Mapping Firewall: 100%|██████████| 6166/6166 [00:00<00:00, 1045650.69it/s]

 > FIREWALL ACTIVE: 1441 unique scaffolds blocked.


In [28]:
# --- 4. PREPARE MONA LOOKUPS ---
# We use the MONA maps to decode the New Candidate set
mona_spec2mol, mona_mol2scaffold, mona_spec2mz = build_lookup_maps(MONA_MOL_DF, MONA_SPEC_DF, "MONA DB")


--- Building Lookups for MONA DB ---
Loading Molecules: /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/mol_df_mona.pkl
Loading Spectra: /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/spec_df_mona.pkl
 > MONA DB Ready: 33398 spectra, 6191 molecules.


In [29]:
# --- 5. FILTER CANDIDATES ---
print(f"\n--- Loading Candidates from {NEW_CANDIDATES} ---")
df_candidates = pd.read_feather(NEW_CANDIDATES)

print("--- Filtering Pairs against Firewall ---")
safe_indices = []

for idx, row in tqdm(df_candidates.iterrows(), total=len(df_candidates), desc="Checking Scaffolds"):
    spec_a = row['name_main']
    spec_b = row['name_sub']
    
    # 1. Get Mol IDs using MONA maps
    mol_a = mona_spec2mol.get(spec_a)
    mol_b = mona_spec2mol.get(spec_b)
    
    if mol_a is None or mol_b is None:
        continue # Skip if metadata missing
        
    # 2. Get Scaffolds using MONA maps
    scaff_a = mona_mol2scaffold.get(mol_a)
    scaff_b = mona_mol2scaffold.get(mol_b)
    
    # 3. Check against ORIGINAL Firewall
    # "Does this new molecule have the same scaffold as an old test molecule?"
    if (scaff_a in forbidden_scaffolds) or (scaff_b in forbidden_scaffolds):
        continue 
        
    safe_indices.append(idx)


--- Loading Candidates from /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/spectra_pairs/balanced_10bin_mona_dataset.feather ---
--- Filtering Pairs against Firewall ---


Checking Scaffolds: 100%|██████████| 808590/808590 [00:32<00:00, 24594.83it/s]


In [30]:
# Create Safe Subset
df_safe = df_candidates.loc[safe_indices].copy()

In [31]:
print(f"\n--- FILTER RESULTS ---")
print(f"Original Pairs: {len(df_candidates)}")
print(f"Safe Pairs:     {len(df_safe)}")
print(f"Rejected:       {len(df_candidates) - len(df_safe)} (Scaffold Leakage)")


--- FILTER RESULTS ---
Original Pairs: 808590
Safe Pairs:     635053
Rejected:       173537 (Scaffold Leakage)


In [32]:
# --- 6. CATEGORIZE AND SAVE ---
print("\n--- Categorizing by Mass Regime ---")

# Calculate Mass Diff using MONA m/z data
if 'mass_difference' not in df_safe.columns:
    print("Calculating mass differences...")
    mz_a = df_safe['name_main'].map(mona_spec2mz)
    mz_b = df_safe['name_sub'].map(mona_spec2mz)
    df_safe['mass_difference'] = abs(mz_a - mz_b)
    # Filter out NaNs if any lookups failed
    df_safe = df_safe.dropna(subset=['mass_difference'])


--- Categorizing by Mass Regime ---
Calculating mass differences...


In [33]:
def assign_regime(mass_diff):
    if mass_diff < 0.01: return "0. Exact Isomers (0 Da)"
    if mass_diff < 1.0: return "1. Isobaric (< 1 Da)"
    if mass_diff < 10.0: return "2. Tiny (1-10 Da)"
    if mass_diff < 50.0: return "3. Medium (10-50 Da)"
    if mass_diff < 100.0: return "4. Large (50-100 Da)"
    return "5. Huge (>100 Da)"

In [34]:
df_safe['Mass_Regime'] = df_safe['mass_difference'].apply(assign_regime)

In [35]:
print("\nSafe Candidates per Regime:")
print(df_safe['Mass_Regime'].value_counts())

df_safe.reset_index(drop=True).to_feather(OUTPUT_PATH)
print(f"\nSaved to: {OUTPUT_PATH}")


Safe Candidates per Regime:
Mass_Regime
5. Huge (>100 Da)          364897
4. Large (50-100 Da)       115522
3. Medium (10-50 Da)       113886
2. Tiny (1-10 Da)           30243
0. Exact Isomers (0 Da)      5518
1. Isobaric (< 1 Da)         4987
Name: count, dtype: int64

Saved to: mona_safe_augmentation_candidates.feather


In [36]:
df_safe.head()

,name_main,name_sub,cosine_similarity,mass_difference,Mass_Regime
0,MoNA_27032,MoNA_78252,0.459223,41.980000,3. Medium (10-50 Da)
1,MoNA_66621,MoNA_92123,0.701804,46.005463,3. Medium (10-50 Da)
2,MoNA_65585,MoNA_83619,0.505656,238.250732,5. Huge (>100 Da)
3,MoNA_75789,MoNA_76066,0.863607,401.193734,5. Huge (>100 Da)
4,MoNA_64797,MoNA_91816,0.104762,276.120880,5. Huge (>100 Da)


## Extract safe pairs from the large dataset

In [17]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

In [18]:
# --- CONFIGURATION ---
BASE_DIR = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data"
SPLIT_DIR = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits"

# 1. Inputs
ORIGINAL_TRAIN = f"{SPLIT_DIR}/stratified_binary_07_dataset_train.feather"
# REPLACE with your large mixed file path
BIG_MIX_SOURCE = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/mass_gate_2_percent/temp_subset_0.02_stratified_binary_07_spec_sim_dataset_train.feather" 

# 2. Metadata (For Firewall & Diversity)
COMBINED_MOL = f"{BASE_DIR}/mol_df_spec_sim.pkl"
COMBINED_SPEC = f"{BASE_DIR}/spec_df_spec_sim.pkl"
VAL_PATH = f"{SPLIT_DIR}/stratified_binary_07_dataset_val.feather"
TEST_PATH = f"{SPLIT_DIR}/stratified_binary_07_dataset_test.feather"

OUTPUT_PATH = f"{SPLIT_DIR}/mass_gate_2_percent/stratified_binary_07_dataset_train_GRID_EQUALIZED.feather"

In [19]:
# 3. Targets
TARGET_PER_CELL = 9000 # Pairs per Sim Bin per Regime

SIM_BINS = [
    (0.0, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0001)
]

In [20]:
# --- 1. SETUP LOOKUPS & FIREWALL ---
print("--- Initializing Maps & Firewall ---")
df_mol = pd.read_pickle(COMBINED_MOL)
df_spec = pd.read_pickle(COMBINED_SPEC)
mol2scaffold = dict(zip(df_mol['mol_id'], df_mol['scaffold']))
spec2mol = dict(zip(df_spec['spec_id'], df_spec['mol_id']))

def get_scaffold(spec_id):
    mol_id = spec2mol.get(spec_id)
    if mol_id is None: return "Unknown"
    return mol2scaffold.get(mol_id, "Unknown")

# Load Forbidden Scaffolds
df_val = pd.read_feather(VAL_PATH)
df_test = pd.read_feather(TEST_PATH)
forbidden_specs = set(df_val['name_main']) | set(df_val['name_sub']) | \
                  set(df_test['name_main']) | set(df_test['name_sub'])

forbidden_scaffolds = set()
for spec_id in tqdm(forbidden_specs, desc="Building Firewall"):
    s = get_scaffold(spec_id)
    if s and s != "Unknown": forbidden_scaffolds.add(s)

print(f" > Firewall Active: {len(forbidden_scaffolds)} scaffolds blocked.")

--- Initializing Maps & Firewall ---


Building Firewall: 100%|██████████| 6166/6166 [00:00<00:00, 1110151.03it/s]

 > Firewall Active: 1441 scaffolds blocked.


In [21]:
# --- 2. LOAD DATASETS ---
print("\n--- Loading Data ---")
df_orig = pd.read_feather(ORIGINAL_TRAIN)
print(f"Original Train: {len(df_orig)}")

print(f"Loading Mixed Source: {BIG_MIX_SOURCE}")
df_mix = pd.read_feather(BIG_MIX_SOURCE) 
if 'label' not in df_mix.columns:
    df_mix['label'] = (df_mix['cosine_similarity'] >= 0.7).astype(int)


--- Loading Data ---
Original Train: 179981
Loading Mixed Source: /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/mass_gate_2_percent/temp_subset_0.02_stratified_binary_07_spec_sim_dataset_train.feather


In [22]:
# --- 3. PREPROCESSING HELPERS ---
def get_regime(row):
    # Use pre-calculated column if exists, else calc
    if 'mass_difference' in row:
        md = row['mass_difference']
    else:
        # Fallback (slow, mostly for safety)
        return "Unknown"
        
    if md < 0.01: return "0. Exact Isomers"
    if md < 1.0: return "1. Isobaric"
    if md < 10.0: return "2. Tiny"
    if md < 50.0: return "3. Medium"
    if md < 100.0: return "4. Large"
    return "5. Huge"

def get_sim_bin(sim):
    for i, (low, high) in enumerate(SIM_BINS):
        if low <= sim < high: return i
    return 4

In [23]:
print("Preprocessing Original...")
if 'mass_difference' not in df_orig.columns:
    # Assuming mass diff exists or you calc it. 
    # If missing, add calc logic here.
    pass 
df_orig['Regime_Norm'] = df_orig.apply(get_regime, axis=1)
df_orig['Sim_Bin_Idx'] = df_orig['cosine_similarity'].apply(get_sim_bin)

print("Preprocessing Mixed Source...")
# Ensure mass diff
if 'mass_difference' not in df_mix.columns:
    spec2mz = dict(zip(df_spec['spec_id'], df_spec['prec_mz']))
    df_mix['mass_difference'] = abs(df_mix['name_main'].map(spec2mz) - df_mix['name_sub'].map(spec2mz))

df_mix['Regime_Norm'] = df_mix.apply(get_regime, axis=1)
df_mix['Sim_Bin_Idx'] = df_mix['cosine_similarity'].apply(get_sim_bin)

Preprocessing Original...
Preprocessing Mixed Source...


In [24]:
# --- 4. DIVERSITY UNDERSAMPLER ---
def undersample_diverse(df_subset, target_n):
    if len(df_subset) <= target_n: return df_subset
    # Group by Scaffold
    scaffolds = [get_scaffold(sid) for sid in df_subset['name_main']]
    df_temp = df_subset.copy()
    df_temp['TEMP_SCAFFOLD'] = scaffolds
    
    groups = df_temp.groupby('TEMP_SCAFFOLD')
    group_keys = list(groups.groups.keys())
    np.random.shuffle(group_keys)
    
    selected_indices = []
    group_iters = [iter(groups.get_group(k).index) for k in group_keys]
    
    while len(selected_indices) < target_n:
        active_iters = []
        for it in group_iters:
            try:
                selected_indices.append(next(it))
                active_iters.append(it)
                if len(selected_indices) >= target_n: break
            except StopIteration: pass
        group_iters = active_iters
        if not group_iters: break
            
    return df_temp.loc[selected_indices].drop(columns=['TEMP_SCAFFOLD'])

In [25]:
# --- 5. EXECUTE GRID EQUALIZER ---
final_dfs = []
regimes = sorted(df_orig['Regime_Norm'].unique())

print(f"\n--- Starting Grid Equalization (Target: {TARGET_PER_CELL}/cell) ---")

for regime in regimes:
    print(f"\n=== Regime: {regime} ===")
    
    for bin_idx, (low, high) in enumerate(SIM_BINS):
        # 1. Get Original Data for this Cell
        subset_orig = df_orig[
            (df_orig['Regime_Norm'] == regime) & 
            (df_orig['Sim_Bin_Idx'] == bin_idx)
        ]
        
        count_orig = len(subset_orig)
        deficit = TARGET_PER_CELL - count_orig
        
        print(f"   [Sim {low}-{high:.1f}] Orig: {count_orig}", end="")
        
        cell_result = None
        
        # CASE A: Original is already too big -> Undersample Diverse
        if deficit < 0:
            print(f" -> Undersampling Original (Diverse)")
            cell_result = undersample_diverse(subset_orig, TARGET_PER_CELL)
            
        # CASE B: Need more -> Fill from Mixed Source
        else:
            print(f" -> Filling {deficit} from Mixed", end="")
            cell_result = subset_orig
            
            if deficit > 0:
                # Filter Mixed Source for this cell
                candidates = df_mix[
                    (df_mix['Regime_Norm'] == regime) & 
                    (df_mix['Sim_Bin_Idx'] == bin_idx)
                ]
                
                # FIREWALL CHECK (Only check candidates we might actually use)
                # This is faster than checking the whole dataset at start
                safe_candidates_indices = []
                needed = deficit
                
                # We shuffle candidates first so we don't just check the first N sorted by ID
                candidates = candidates.sample(frac=1.0, random_state=42)
                
                for idx, row in candidates.iterrows():
                    sa = get_scaffold(row['name_main'])
                    sb = get_scaffold(row['name_sub'])
                    
                    if sa not in forbidden_scaffolds and sb not in forbidden_scaffolds:
                        safe_candidates_indices.append(idx)
                        if len(safe_candidates_indices) >= needed:
                            break
                
                # Add Safe Candidates
                if safe_candidates_indices:
                    added = candidates.loc[safe_candidates_indices]
                    # Align columns
                    common_cols = list(set(df_orig.columns) & set(added.columns))
                    added = added[common_cols]
                    cell_result = pd.concat([cell_result, added])
                    print(f" (Found {len(added)} safe)")
                else:
                    print(f" (No safe candidates found)")
            else:
                print("")

        final_dfs.append(cell_result)


--- Starting Grid Equalization (Target: 9000/cell) ---

=== Regime: 0. Exact Isomers ===
   [Sim 0.0-0.2] Orig: 27 -> Filling 8973 from Mixed (Found 220 safe)
   [Sim 0.2-0.4] Orig: 138 -> Filling 8862 from Mixed (Found 531 safe)
   [Sim 0.4-0.6] Orig: 436 -> Filling 8564 from Mixed (Found 1187 safe)
   [Sim 0.6-0.8] Orig: 2250 -> Filling 6750 from Mixed (Found 4269 safe)
   [Sim 0.8-1.0] Orig: 7451 -> Filling 1549 from Mixed (Found 1549 safe)

=== Regime: 1. Isobaric ===
   [Sim 0.0-0.2] Orig: 147 -> Filling 8853 from Mixed (Found 994 safe)
   [Sim 0.2-0.4] Orig: 219 -> Filling 8781 from Mixed (Found 1383 safe)
   [Sim 0.4-0.6] Orig: 285 -> Filling 8715 from Mixed (Found 1672 safe)
   [Sim 0.6-0.8] Orig: 642 -> Filling 8358 from Mixed (Found 2369 safe)
   [Sim 0.8-1.0] Orig: 910 -> Filling 8090 from Mixed (Found 3736 safe)

=== Regime: 2. Tiny ===
   [Sim 0.0-0.2] Orig: 1306 -> Filling 7694 from Mixed (Found 7694 safe)
   [Sim 0.2-0.4] Orig: 1373 -> Filling 7627 from Mixed (Found 762

In [ ]:
# --- 6. SAVE ---
print("\n--- Finalizing ---")
df_final = pd.concat(final_dfs, axis=0, ignore_index=True)
df_final = df_final.sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f"Total Dataset Size: {len(df_final)}")
print("\nFinal Counts per Regime:")
print(df_final['Regime_Norm'].value_counts())

df_final.to_feather(OUTPUT_PATH)
print(f"Saved to: {OUTPUT_PATH}")


--- Finalizing ---
Total Dataset Size: 210415

Final Counts per Regime:
Regime_Norm
3. Medium           45000
5. Huge             45000
2. Tiny             45000
4. Large            45000
0. Exact Isomers    18058
1. Isobaric         12357
Name: count, dtype: int64
Saved to: /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/mass_gate_2_percent/stratified_binary_07_dataset_train_GRID_EQUALIZED.feather


: 